In [1]:
%cd ../../../

/path/to/repo/my-coles-gnn-experimetns/scenario_age_pred


/path/to/repo/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [24]:
from collections import defaultdict
import os

BATCH_SIZE = 40
MAX_EPOCHES_OPTIONS = range(10, 160, 10) 
SPLIT_COUNT = 2


MODEL_EPOCH_MAP = {
    # "wl-0.1_gnn-GAT_residual-True_weight_decay-0": [11], Убрал, потому что нет hydra output'а
    "wl-0.5_gnn-GAT_residual-True_weight_decay-0": [11, 31, 51, 71, 91, 111, 131, 171, 191],
    "wl-0.5_gnn-GAT_residual-True_weight_decay-0.01": [11],
    "wl-0.5_gnn-GraphSAGE_residual-True_weight_decay-0": [11, 31, 51, 71, 91, 111, 131, 171]
}




expected_experiments = []

for gnn_subexperiment_name, gnn_pretrain_epochs_options in MODEL_EPOCH_MAP.items():
    for pretrain_epoch in gnn_pretrain_epochs_options:
        for train_epochs in MAX_EPOCHES_OPTIONS:
            if train_epochs == 150:
                experiment_name=f"coles_only__item_id_embed_init_with_pretrained_gnn__bs_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{gnn_subexperiment_name}__pretrain_epoches_{pretrain_epoch}__epoches_{train_epochs}"
            else:
                experiment_name=f"coles_only__item_id_embed_init_with_pretrained_gnn__bs_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{gnn_subexperiment_name}__pretrain_epoches_{pretrain_epoch}__{train_epochs}_epoches"
            expected_experiments.append(experiment_name)


In [25]:
all_hydra_outputs = os.listdir('hydra_outputs')

hydra_outputs_that_dont_exist = []
train_epochs = 150

for gnn_subexperiment_name, gnn_pretrain_epochs_options in MODEL_EPOCH_MAP.items():
    for pretrain_epoch in gnn_pretrain_epochs_options:
        experiment_name=f"coles_only__item_id_embed_init_with_pretrained_gnn__bs_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{gnn_subexperiment_name}__pretrain_epoches_{pretrain_epoch}__epoches_{train_epochs}"
        if experiment_name not in all_hydra_outputs:
            hydra_outputs_that_dont_exist.append(experiment_name)


In [26]:
hydra_outputs_that_dont_exist

[]

In [27]:
expected_experiments[:10]

['coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__10_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__20_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__30_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__40_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__50_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__60_epoches',
 'coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_cou

In [28]:
import sys, os; sys.path.append(os.path.abspath( '..'))

In [29]:
from omegaconf import OmegaConf 

In [30]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

In [31]:
from ptls_extension_2024_research.latex_table_creation.latex_table_creation import create_latex_table, get_metrics
from ptls_extension_2024_research.latex_table_creation.experiment_dicts_list_modifiers import bolden_top_k, sort_by_col
from ptls_extension_2024_research.latex_table_creation.prefix_map import get_idxs_where_all_metrics_superpass, prefix_map_from_idx_lst

In [32]:
with open("results/scenario_age_coles__item_id_embedding_init_with_gnn_embs__supercomputer.txt") as f:
    report_file_content = f.read()

In [33]:
COLES_METRICS = {r"\textbf{AUC test}": 0.877, r"\textbf{ACC test}": 0.793}

In [43]:
from typing import Optional
import re

# Some information is present in name only, everything else is 
# suposed to be retrived from the config (yaml read wit OmegaConf).
# For a uniformal interface both config and experiment_name are always passed to hyperparam getters.


def get_batchsize(config, experiment_name) ->  int:
    return int(config['data_module']['train_batch_size'])


def get_gnn_loss_alpha(config, experiment_name) -> Optional[float]:
    matches = re.search(r'wl-(\d+\.\d+)', experiment_name)
    if matches is None:
        return None
    return float(matches.group(1))


def get_gnn_task(config, experiment_name) -> str:
    if 'GRACE' in experiment_name:
        return 'GRACE'
    return 'Link and weight prediction'


def get_has_residual(config, experiment_name) -> str:
    has_residual_str = re.search(r'residual-(true|True|False)', experiment_name).group(1)
    if  has_residual_str in ['true', 'True']:
        return True
    return False


def get_split_count(config, experiment_name) -> int:
    return int(config['data_module']['train_data']['splitter']['split_count'])


import re

def get_train_epoches(config, experiment_name) -> int:
    # Look for 'epoches_<number>' but exclude 'pretrain_epoches_<number>'
    v1 = re.search(r'(?<!pretrain_)epoches_(\d+)', experiment_name)
    if v1 is not None:
        return int(v1.group(1))
    
    # Look for '<number>_epoches' but exclude 'pretrain_<number>_epoches'
    v2 = re.search(r'(?<!pretrain_)(\d+)_epoches', experiment_name)
    if v2 is not None:
        return int(v2.group(1))
    
    return None


def get_weight_decay(config, experiment_name):
    return re.search(r'weight_decay-(\d*\.?\d+)', experiment_name).group(1)

hyperparams_to_getters = {
    r"\textbf{GNN Loss alpha}": get_gnn_loss_alpha,
    r"\textbf{GNN task}": get_gnn_task,
    r"\textbf{GNN}": lambda config, experiment_name: re.search(r'gnn-(GraphSAGE|GAT)', experiment_name).group(1),
    r"\textbf{Has residual connection}": get_has_residual,
    r"\textbf{Batch size}": get_batchsize,
    r"\textbf{sc}": get_split_count,
    r"\textbf{weight decay}": get_weight_decay,
    r"\textbf{Pretrain epoches}": lambda config, experiment_name: int(re.search(r'pretrain_epoches_(\d+)', experiment_name).group(1)),
    r"\textbf{Train epoches}": get_train_epoches,

}

In [44]:
experiment_dicts_list = []

for experiment_name in expected_experiments:
    hydra_config_path = os.path.join('hydra_outputs', experiment_name, '.hydra', 'config.yaml')
    hydra_config_path = re.sub(r'\d+_epoches', 'epoches_150', hydra_config_path)
    config = OmegaConf.load(hydra_config_path)
    hyperparams = {k: v(config, experiment_name) for k, v in hyperparams_to_getters.items()}
    metrics = get_metrics(report_file_content, experiment_name, {'accuracy': r"\textbf{ACC test}"})
    if metrics is not None:
        experiment_dicts_list.append({**hyperparams, **metrics})

Could not find experiment coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__100_epoches in file content
Could not find experiment coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__110_epoches in file content
Could not find experiment coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__120_epoches in file content
Could not find experiment coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__130_epoches in file content
Could not find experiment coles_only__item_id_embed_init_with_pretrained_gnn__bs_40__split_count_2__wl-0.5_gnn-GAT_residual-True_weight_decay-0__pretrain_epoches_11__140_epoches in file content
Could not find experiment cole

In [45]:
WEIGHTS_TYPE = 'type 2'

# experiment_dicts_list stores raw values

# experiment_dicts_list stores raw values, list is sorted
experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{ACC test}")
# prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")

# experiment_dicts_list stores strings
experiment_dicts_of_str_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles only with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_of_str_list
    }
}



In [46]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = "Coles embedding layer is initialized with pretrained gnn. Scenario age"

row_prefix_dict = None

# row_prefix_dict = {
#     EXPERIMENT_NAME: {
#         WEIGHTS_TYPE: prefix_map
#     }
# }

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN Loss alpha} & \textbf{GNN task} & \textbf{GNN} & \textbf{Has residual connection} & \textbf{Batch size} & \textbf{sc} & \textbf{weight decay} & \textbf{Pretrain epoches} & \textbf{Train epoches} & \textbf{ACC test}\\
\hline
\multirow{113}{*}{\makecell{Coles only with bpr}} &\multirow{113}{*}{type 2} & 0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0 & 131 & 150 & \textbf{0.638} \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0.01 & 11 & 150 & \textbf{0.637} \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GAT & True & 40 & 2 & 0 & 31 & 150 & \textbf{0.636} \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GraphSAGE & True & 40 & 2 & 0 & 31 & 150 & 0.636 \\ \cline{3-12} 
& &0.5 & Link and weight prediction & GraphSAGE & True & 40 & 2 & 0 & 171 & 150 & 0.636 \\ \cline{3-12} 
& 